In [ ]:
pip-installs ruptures

In [ ]:
# =============================================================
# DARKLINE — NOTEBOOK A : EXTRACTION
# Team CORTEX · Accenture Innovation Challenge 2026
# =============================================================
# This notebook does the ONE expensive job: turning the 1156-column
# date file into one first-touch timestamp per station per part.
# Everything else happens in Notebook B.
#
# Every stage checkpoints to /kaggle/working. If a stage has already
# run, it is skipped. You can re-run this notebook top to bottom at
# any time and it will only do the work that is still missing.
# =============================================================

import os, re, gc, json, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

BASE = Path('/kaggle/input/bosch-production-line-performance')
if not BASE.exists():
    BASE = Path('/kaggle/input/competitions/bosch-production-line-performance')
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True, parents=True)

print("Input path:", BASE)
print("Files present:")
for f in sorted(BASE.glob('*')):
    print(f"   {f.name:35s} {f.stat().st_size/1024/1024:8.1f} MB")


def ck(name):
    """Path of a checkpoint file."""
    return WORK / f"{name}.parquet"


def have(name):
    """True if this stage already finished."""
    p = ck(name)
    ok = p.exists() and p.stat().st_size > 0
    print(f"[checkpoint] {name:28s} {'FOUND — will skip' if ok else 'missing — will build'}")
    return ok


def save(df, name):
    p = ck(name)
    df.to_parquet(p, index=False)
    print(f"[saved] {name}  shape={df.shape}  size={p.stat().st_size/1024/1024:.1f} MB")
    return p


def load(name):
    return pd.read_parquet(ck(name))


class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); print(f"\n>>> {self.label} ...", flush=True); return self
    def __exit__(self, *a): print(f"<<< {self.label} done in {time.time()-self.t:.1f}s", flush=True)


print("\nSetup complete.")

In [ ]:
# =============================================================
# STAGE 1 — Read ONLY the column headers (cheap, seconds) and work
# out the structure of the line: which stations exist, and which of
# them are measured versus merely timed.
#
# THIS IS THE MOST IMPORTANT CELL IN THE PROJECT. The number it
# prints is the factual basis of our entire pitch.
# =============================================================

NUM = BASE / 'train_numeric.csv.zip'
DAT = BASE / 'train_date.csv.zip'
CAT = BASE / 'train_categorical.csv.zip'

date_cols = pd.read_csv(DAT, nrows=0).columns.tolist()
num_cols  = pd.read_csv(NUM, nrows=0).columns.tolist()
cat_cols  = pd.read_csv(CAT, nrows=0).columns.tolist()

PAT = re.compile(r'^L(\d+)_S(\d+)_[DF](\d+)$')

def station_of(col):
    m = PAT.match(col)
    return f"L{m.group(1)}_S{m.group(2)}" if m else None

date_by_station = defaultdict(list)
num_by_station  = defaultdict(list)
cat_by_station  = defaultdict(list)

for c in date_cols:
    s = station_of(c)
    if s: date_by_station[s].append(c)
for c in num_cols:
    if c in ('Id', 'Response'): continue
    s = station_of(c)
    if s: num_by_station[s].append(c)
for c in cat_cols:
    if c == 'Id': continue
    s = station_of(c)
    if s: cat_by_station[s].append(c)

def skey(s):
    l, st = s.split('_')
    return (int(l[1:]), int(st[1:]))

stations = sorted(date_by_station.keys(), key=skey)

catalog = pd.DataFrame({
    'station':   stations,
    'line':      [int(s.split('_')[0][1:]) for s in stations],
    'station_no':[int(s.split('_')[1][1:]) for s in stations],
    'n_date':    [len(date_by_station[s]) for s in stations],
    'n_numeric': [len(num_by_station.get(s, [])) for s in stations],
    'n_categorical':[len(cat_by_station.get(s, [])) for s in stations],
})

# ---- THE DARK-STATION DEFINITION -----------------------------
# A station that is TIMED (it has date columns, so we know a part
# passed through it and when) but NOT MEASURED (no numeric feature
# columns, so we have no idea what happened to the part there).
catalog['is_dark'] = catalog['n_numeric'] == 0
catalog['instrumentation'] = np.where(catalog['is_dark'], 'DARK', 'MEASURED')

n_dark = int(catalog['is_dark'].sum())
n_tot  = len(catalog)

print("="*66)
print("STATION CATALOG")
print("="*66)
print(catalog.to_string(index=False))
print("="*66)
print(f"Total stations timed in train_date : {n_tot}")
print(f"DARK stations (0 numeric features) : {n_dark}")
print(f"Fraction of the line that is dark  : {n_dark/n_tot*100:.1f}%")
print("="*66)
print()
print(">>> SEND THIS NUMBER TO KARTIK. If it is not ~35%, the slides")
print(">>> change to match the data. The data wins. Never the other way.")

save(catalog, 'station_catalog')

with open(WORK / 'column_map.json', 'w') as f:
    json.dump({
        'date_by_station': {k: v for k, v in date_by_station.items()},
        'num_by_station':  {k: v for k, v in num_by_station.items()},
        'cat_by_station':  {k: v for k, v in cat_by_station.items()},
        'stations': stations,
    }, f)
print("[saved] column_map.json")

In [ ]:
# =============================================================
# STAGE 2 — THE ONLY SLOW CELL IN THE PROJECT (~15-30 minutes).
#
# train_date.csv.zip is 1.18M rows x 1157 columns. We collapse it
# to 1.18M rows x ~52 columns: for each part, the FIRST timestamp
# observed at each station.
#
# CRASH PROTECTION: each chunk is written to disk as it completes.
# If the kernel dies at chunk 17 of 24, re-running this cell picks
# up at chunk 17. Nothing is recomputed.
# =============================================================

CHUNK = 50_000
PARTS_DIR = WORK / 'date_parts'
PARTS_DIR.mkdir(exist_ok=True)

if have('station_times'):
    station_times = load('station_times')
    print("Loaded existing station_times, shape:", station_times.shape)
else:
    with open(WORK / 'column_map.json') as f:
        cmap = json.load(f)
    date_by_station = cmap['date_by_station']
    stations = cmap['stations']

    # float32 halves memory versus the pandas default of float64
    dtypes = {c: np.float32 for c in
              [c for c in pd.read_csv(DAT, nrows=0).columns if c != 'Id']}
    dtypes['Id'] = np.int32

    done_chunks = {int(p.stem.split('_')[1]) for p in PARTS_DIR.glob('chunk_*.parquet')}
    if done_chunks:
        print(f"Resuming — {len(done_chunks)} chunks already on disk: "
              f"up to chunk {max(done_chunks)}")

    with Timer("Reducing train_date to per-station timestamps"):
        reader = pd.read_csv(DAT, chunksize=CHUNK, dtype=dtypes)
        for i, chunk in enumerate(reader):
            if i in done_chunks:
                continue
            out = {'Id': chunk['Id'].values}
            for s in stations:
                cols = date_by_station[s]
                # nanmin across that station's date columns = first touch
                block = chunk[cols].to_numpy(dtype=np.float32, copy=False)
                with np.errstate(all='ignore'):
                    out[s] = np.nanmin(block, axis=1)
            pd.DataFrame(out).to_parquet(
                PARTS_DIR / f'chunk_{i:04d}.parquet', index=False)
            del chunk, out
            gc.collect()
            print(f"   chunk {i:3d} written  ({(i+1)*CHUNK:,} rows seen)", flush=True)

    with Timer("Concatenating chunks"):
        files = sorted(PARTS_DIR.glob('chunk_*.parquet'))
        station_times = pd.concat(
            [pd.read_parquet(f) for f in files], ignore_index=True)
        for c in station_times.columns:
            if c != 'Id':
                station_times[c] = station_times[c].astype(np.float32)

    save(station_times, 'station_times')

print(station_times.shape)
station_times.head()

In [ ]:
# =============================================================
# STAGE 3 — Turn per-station timestamps into per-part facts.
# Fully vectorised with numpy. No .apply anywhere — a row-wise
# apply over 1.18M rows takes hours and will time the kernel out.
# =============================================================

if have('parts'):
    parts = load('parts')
else:
    with open(WORK / 'column_map.json') as f:
        stations = json.load(f)['stations']

    with Timer("Building part-level features"):
        M = station_times[stations].to_numpy(dtype=np.float32)   # (n_parts, n_stations)
        ids = station_times['Id'].to_numpy()
        visited = ~np.isnan(M)

        with np.errstate(all='ignore'):
            first_ts = np.nanmin(M, axis=1)
            last_ts  = np.nanmax(M, axis=1)

        parts = pd.DataFrame({
            'Id': ids,
            'first_timestamp': first_ts,
            'last_timestamp':  last_ts,
            'total_cycle_time': last_ts - first_ts,
            'n_stations_visited': visited.sum(axis=1).astype(np.int16),
        })

    with Timer("Building path signatures (vectorised)"):
        # A part's path = the stations it visited, in timestamp order.
        # Encode as a compact string so identical routes group together.
        order = np.argsort(np.where(visited, M, np.inf), axis=1)
        codes = np.array([s.split('_S')[1] for s in stations])
        sigs = []
        nvis = visited.sum(axis=1)
        for r in range(M.shape[0]):
            k = nvis[r]
            sigs.append('-'.join(codes[order[r, :k]]))
            if r % 200_000 == 0:
                print(f"   {r:,} / {M.shape[0]:,}", flush=True)
        parts['path_signature'] = sigs

    # ---- attach the label ------------------------------------
    resp = pd.read_csv(NUM, usecols=['Id', 'Response'],
                       dtype={'Id': np.int32, 'Response': np.int8})
    parts = parts.merge(resp, on='Id', how='left')

    # ---- THE TEMPORAL SPLIT ----------------------------------
    # Manufacturing data is time-ordered. A random split leaks the
    # future into training. We split by TIME: earliest 70% train,
    # next 15% validation, last 15% test.
    parts = parts.sort_values('first_timestamp',
                              kind='mergesort').reset_index(drop=True)
    n = len(parts)
    i70, i85 = int(n * 0.70), int(n * 0.85)
    parts['split'] = 'train'
    parts.loc[i70:i85, 'split'] = 'val'
    parts.loc[i85:,   'split'] = 'test'

    save(parts, 'parts')

print(parts['split'].value_counts())
print()
print("Failure rate by split:")
print(parts.groupby('split')['Response'].agg(['mean', 'sum', 'count']))
print()
print("Split time boundaries (anonymised Bosch time units):")
for s in ['train', 'val', 'test']:
    sub = parts[parts.split == s]
    print(f"  {s:6s} {sub.first_timestamp.min():9.2f} -> {sub.first_timestamp.max():9.2f}")

print()
print("Top 10 routing paths by share:")
top_paths = parts['path_signature'].value_counts(normalize=True).head(10)
print((top_paths * 100).round(2).to_string())

print()
print("Failure rate by top path  <-- watch for variation, this is a real finding")
pr = parts.groupby('path_signature').agg(
    n=('Response', 'size'), fail_rate=('Response', 'mean'))
pr = pr[pr.n > 5000].sort_values('fail_rate', ascending=False)
print((pr.head(12).assign(fail_rate=lambda d: (d.fail_rate*100).round(3))).to_string())

In [ ]:
# =============================================================
# STAGE 4 — Free the chunk files and confirm the handoff artifacts.
# =============================================================

import shutil
if PARTS_DIR.exists():
    shutil.rmtree(PARTS_DIR)      # chunk files no longer needed
    print("Removed intermediate chunk directory.")

print("\nFiles that Notebook B will consume:")
for f in sorted(WORK.glob('*')):
    if f.is_file():
        print(f"   {f.name:32s} {f.stat().st_size/1024/1024:8.1f} MB")

print()
print("="*66)
print("NOTEBOOK A COMPLETE")
print("="*66)
print("NOW DO THIS, IN THIS ORDER:")
print("  1. Top right -> Save Version -> 'Save & Run All (Commit)'")
print("  2. Wait for it to finish (it re-runs, but every stage is")
print("     checkpointed so it is fast the second time).")
print("  3. The files above become this notebook's OUTPUT.")
print("  4. In Notebook B: Add Data -> Your Work -> pick this")
print("     notebook's output.")
print("="*66)